# Train a Wake Word Detection Model: "Hey Asterix"

This notebook demonstrates the complete pipeline for training a **keyword spotting (KWS)** model targeting TensorFlow Lite for Microcontrollers (TFLM). The model detects **"Hey Asterix"** on embedded hardware such as ESP32-S3 or STM32F7.

## Pipeline
```
Audio Data -> MFCC Features -> DS-CNN Model -> INT8 Quantize -> C Array -> MCU Firmware
```

**Key design decisions:**
- Input: 2-second audio @ 16 kHz, 40-band MFCC (~197 frames x 40 coefficients)
- Architecture: DS-CNN (Depthwise Separable CNN) -- MCU-optimal, no dynamic allocation
- Quantization: Full INT8, ~20-40 KB Flash, ~30-80 KB RAM (tensor arena)

<table class="tfo-notebook-buttons" align="left">
  <td><a target="_blank" href="https://colab.research.google.com/github/Tdieney/TinyML-learning/blob/main/notebooks/02_wake_word_hey_asterix/train_hey_asterix_model.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a></td>
  <td><a target="_blank" href="https://github.com/Tdieney/TinyML-learning/blob/main/notebooks/02_wake_word_hey_asterix/train_hey_asterix_model.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View on GitHub</a></td>
</table>

## Configure Defaults

In [ ]:
import os

MODELS_DIR = 'models/02_hey_asterix/'
os.makedirs(MODELS_DIR, exist_ok=True)
MODEL_SAVED  = MODELS_DIR + 'model_saved'
MODEL_TFLITE = MODELS_DIR + 'model.tflite'
MODEL_INT8   = MODELS_DIR + 'model_int8.tflite'
MODEL_CC     = MODELS_DIR + 'model.cc'

SAMPLE_RATE   = 16000
CLIP_DURATION = 2.0
NUM_SAMPLES   = int(SAMPLE_RATE * CLIP_DURATION)
N_MFCC     = 40
N_FFT      = 512
HOP_LENGTH = int(SAMPLE_RATE * 0.010)
WIN_LENGTH = int(SAMPLE_RATE * 0.025)
N_MELS     = 40
TIME_FRAMES = 1 + (NUM_SAMPLES - WIN_LENGTH) // HOP_LENGTH
INPUT_SHAPE = (TIME_FRAMES, N_MFCC, 1)

BATCH_SIZE    = 64
EPOCHS        = 50
LEARNING_RATE = 1e-3
TRAIN_SPLIT   = 0.80
VAL_SPLIT     = 0.10
TEST_SPLIT    = 0.10
SEED = 42

print(f'Input shape : {INPUT_SHAPE}  ({TIME_FRAMES} frames x {N_MFCC} MFCCs x 1 ch)')
print(f'Models dir  : {os.path.abspath(MODELS_DIR)}')

## Setup Environment

### Install Dependencies

In [ ]:
!pip install -q tensorflow ai-edge-litert
!pip install -q librosa soundfile gTTS pydub
!pip install -q tqdm scikit-learn matplotlib seaborn
!apt-get -qq install -y ffmpeg 2>/dev/null || true
print('All dependencies installed.')

### Import Dependencies

In [ ]:
import os, random, tarfile, urllib.request, io, re, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa, librosa.display
import soundfile as sf
from tqdm import tqdm
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import ai_edge_litert.interpreter as litert

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')

## Dataset

| Class | Label | Description |
|-------|-------|-------------|
| **Positive** | 1 | Audio: "Hey Asterix" (2 s @ 16 kHz) |
| **Negative** | 0 | Other speech / background noise (2 s @ 16 kHz) |

### Positive Sample Strategy (inspired by OpenWakeWord)
We stack **three TTS engines** for maximum acoustic diversity:
- **gTTS** (Google TTS, internet) -- consistent American English
- **edge-tts** (Microsoft Azure Neural TTS, internet) -- 8+ EN voices across gender/accent
- **Coqui TTS** (offline VCTK multi-speaker) -- 100+ studio-recorded speakers

All clips pass through `augment()`: pitch shift +-3 st, time stretch +-20%, additive noise.

You can also **upload real recordings** of "Hey Asterix" in the optional cell.

**Negative**: Speech Commands v2 + background noise (padded/trimmed to 2 s).

> **Production tip**: Record >=500 real utterances from multiple speakers and environments.

### 1. Download Google Speech Commands Dataset (Negative Samples)

In [ ]:
DATA_DIR  = Path('/tmp/speech_commands')
AUDIO_DIR = Path('/tmp/hey_asterix_audio')
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
(AUDIO_DIR / 'positive').mkdir(exist_ok=True)
(AUDIO_DIR / 'negative').mkdir(exist_ok=True)

SPEECH_URL = 'http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz'
TARBALL    = '/tmp/speech_commands_v0.02.tar.gz'

if not DATA_DIR.exists():
    print('Downloading Google Speech Commands v2 (~2.3 GB) ...')
    urllib.request.urlretrieve(SPEECH_URL, TARBALL)
    with tarfile.open(TARBALL, 'r:gz') as tar:
        tar.extractall(DATA_DIR)
    print('Done.')
else:
    print('Dataset already present.')

word_dirs = sorted([d.name for d in DATA_DIR.iterdir()
                    if d.is_dir() and not d.name.startswith('_')])
print(f'\nWord categories ({len(word_dirs)}): {word_dirs}')

### 2. Collect Negative Samples

In [ ]:
import shutil

neg_files = []
for word in word_dirs[:20]:
    neg_files.extend(list((DATA_DIR / word).glob('*.wav'))[:150])

noise_dir = DATA_DIR / '_background_noise_'
if noise_dir.exists():
    for nf in noise_dir.glob('*.wav'):
        audio, _ = librosa.load(str(nf), sr=SAMPLE_RATE, mono=True)
        for i in range(min(len(audio)//NUM_SAMPLES, 50)):
            chunk = audio[i*NUM_SAMPLES:(i+1)*NUM_SAMPLES]
            sf.write(str(AUDIO_DIR/'negative'/f'noise_{nf.stem}_{i:03d}.wav'), chunk, SAMPLE_RATE)

for i, src in enumerate(tqdm(neg_files, desc='Copying negatives')):
    shutil.copy2(src, AUDIO_DIR/'negative'/f'neg_{i:05d}.wav')

print(f'\nNegative samples: {len(list((AUDIO_DIR/"negative").glob("*.wav")))}')

### 3a. Shared Augmentation Helpers

All TTS engines share the same `augment()` pipeline:
- **Pitch shift**: +-3 semitones (random)
- **Time stretch**: +-20% rate (random)
- **Centre pad/trim** to exactly `NUM_SAMPLES` samples
- **Additive Gaussian noise** at SNR ~20 dB
- **Random level jitter** 70-100% of peak

In [ ]:
from pydub import AudioSegment

AUG_PER_CLIP = 8
PHRASES = ["Hey Asterix", "Hey, Asterix", "Hey Asterix!", "hey asterix", "Hey Asterix."]


def _pad_trim(audio):
    """Centre-pad or trim waveform to exactly NUM_SAMPLES."""
    if len(audio) < NUM_SAMPLES:
        p = NUM_SAMPLES - len(audio)
        audio = np.pad(audio, (p // 2, p - p // 2))
    else:
        s = (len(audio) - NUM_SAMPLES) // 2
        audio = audio[s:s + NUM_SAMPLES]
    return audio


def augment(audio, sr, seed):
    """Pitch shift, time stretch, noise, level jitter -- output is NUM_SAMPLES long."""
    rng = np.random.default_rng(seed)
    audio = librosa.effects.pitch_shift(audio, sr=sr, n_steps=rng.uniform(-3.0, 3.0))
    audio = librosa.effects.time_stretch(audio, rate=rng.uniform(0.80, 1.20))
    audio = _pad_trim(audio)
    audio = audio + rng.uniform(0.001, 0.005) * rng.standard_normal(len(audio))
    pk = np.max(np.abs(audio))
    if pk > 0:
        audio = audio / pk * rng.uniform(0.7, 1.0)
    return audio.astype(np.float32)


def _pydub_to_array(seg):
    """Convert pydub AudioSegment to float32 numpy at SAMPLE_RATE."""
    seg = seg.set_frame_rate(SAMPLE_RATE).set_channels(1).set_sample_width(2)
    return np.array(seg.get_array_of_samples(), dtype=np.float32) / 32768.0


pos_idx = 0
print(f"AUG_PER_CLIP={AUG_PER_CLIP}, CLIP_DURATION={CLIP_DURATION}s, NUM_SAMPLES={NUM_SAMPLES}")

### 3b. Engine 1 -- gTTS (Google TTS, internet required)

Google TTS: single consistent American English voice. Phrase variants create subtle prosody differences.

In [ ]:
from gtts import gTTS

N_GTTS = 30  # base clips -> N_GTTS x AUG_PER_CLIP augmented positives


def gtts_to_array(text):
    tts = gTTS(text=text, lang="en", slow=False)
    buf = io.BytesIO()
    tts.write_to_fp(buf)
    buf.seek(0)
    return _pydub_to_array(AudioSegment.from_mp3(buf))


for i in tqdm(range(N_GTTS), desc="gTTS"):
    try:
        base = _pad_trim(gtts_to_array(PHRASES[i % len(PHRASES)]))
        for j in range(AUG_PER_CLIP):
            sf.write(str(AUDIO_DIR / "positive" / f"pos_{pos_idx:05d}.wav"),
                     augment(base, SAMPLE_RATE, i * 1000 + j), SAMPLE_RATE)
            pos_idx += 1
    except Exception as e:
        print(f"  [gTTS] clip {i}: {e}")

print(f"After gTTS: {pos_idx} positive clips")

### 3c. Engine 2 -- edge-tts (Microsoft Azure Neural TTS, internet required)

24+ English Neural voices across gender, accent, and speaking style. Far more diverse than a single gTTS voice.

In [ ]:
!pip install -q edge-tts
import asyncio, edge_tts

EDGE_VOICES = [
    "en-US-AriaNeural",    # female, conversational
    "en-US-GuyNeural",     # male, professional
    "en-US-JennyNeural",   # female, friendly
    "en-GB-SoniaNeural",   # female, British
    "en-GB-RyanNeural",    # male, British
    "en-AU-NatashaNeural", # female, Australian
    "en-AU-WilliamNeural", # male, Australian
    "en-IN-NeerjaNeural",  # female, Indian English
]
N_EDGE = 30


async def _edge_bytes(text, voice):
    communicate = edge_tts.Communicate(text, voice)
    chunks = []
    async for chunk in communicate.stream():
        if chunk["type"] == "audio":
            chunks.append(chunk["data"])
    return b"".join(chunks)


def edge_to_array(text, voice):
    mp3 = asyncio.get_event_loop().run_until_complete(_edge_bytes(text, voice))
    return _pydub_to_array(AudioSegment.from_mp3(io.BytesIO(mp3)))


for i in tqdm(range(N_EDGE), desc="edge-tts"):
    voice  = EDGE_VOICES[i % len(EDGE_VOICES)]
    phrase = PHRASES[i   % len(PHRASES)]
    try:
        base = _pad_trim(edge_to_array(phrase, voice))
        for j in range(AUG_PER_CLIP):
            sf.write(str(AUDIO_DIR / "positive" / f"pos_{pos_idx:05d}.wav"),
                     augment(base, SAMPLE_RATE, i * 2000 + j), SAMPLE_RATE)
            pos_idx += 1
    except Exception as e:
        print(f"  [edge-tts] clip {i} voice={voice}: {e}")

print(f"After edge-tts: {pos_idx} positive clips")

### 3d. Engine 3 -- Coqui TTS (offline, VCTK multi-speaker, runs on Colab CPU)

Coqui TTS: VCTK multi-speaker VITS model with 100+ unique studio speakers.
Each speaker has a distinct F0 range, vocal tract length, and speaking rate.
This is the closest approach to OpenWakeWord's speaker-diversity strategy.

> **First run only**: VCTK checkpoint (~400 MB) downloads to `/root/.local/share/tts/`.

In [ ]:
!pip install -q TTS
from TTS.api import TTS as CoquiTTS

COQUI_MODEL = "tts_models/en/vctk/vits"  # 100+ studio speakers
N_COQUI = 40

print("Loading Coqui TTS model (downloads ~400 MB on first run) ...")
coqui = CoquiTTS(COQUI_MODEL, gpu=False)
all_spk = coqui.speakers
print(f"Available VCTK speakers: {len(all_spk)}  (sample: {all_spk[:5]})")

step = max(1, len(all_spk) // N_COQUI)
sel_spk = all_spk[::step][:N_COQUI]


def coqui_to_array(text, speaker):
    """Synthesise with Coqui VCTK speaker; resample to SAMPLE_RATE if needed."""
    wav = coqui.tts(text=text, speaker=speaker)
    audio = np.array(wav, dtype=np.float32)
    model_sr = coqui.synthesizer.output_sample_rate
    if model_sr != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=model_sr, target_sr=SAMPLE_RATE)
    return audio


for i, spk in enumerate(tqdm(sel_spk, desc="Coqui TTS")):
    phrase = PHRASES[i % len(PHRASES)]
    try:
        base = _pad_trim(coqui_to_array(phrase, spk))
        for j in range(AUG_PER_CLIP):
            sf.write(str(AUDIO_DIR / "positive" / f"pos_{pos_idx:05d}.wav"),
                     augment(base, SAMPLE_RATE, i * 3000 + j), SAMPLE_RATE)
            pos_idx += 1
    except Exception as e:
        print(f"  [Coqui] speaker={spk}: {e}")

print(f"After Coqui TTS: {pos_idx} positive clips")

### 3e. (Optional) Upload Your Own Real Recordings

Real microphone recordings are the single biggest quality boost.
Upload `.wav` or `.mp3` files of yourself saying **"Hey Asterix"**.
Each clip is augmented `AUG_PER_CLIP` times.

**Requirements**: any sample rate (auto-resampled), 1-3 s recommended.

In [ ]:
# OPTIONAL: close the file dialog to skip.
from google.colab import files as colab_files

print('Upload WAV/MP3 of "Hey Asterix", or close dialog to skip.')
try:
    uploaded = colab_files.upload()
except Exception:
    uploaded = {}

for filename, data in uploaded.items():
    tmp = f"/tmp/user_{filename}"
    with open(tmp, "wb") as f:
        f.write(data)
    try:
        raw, sr0 = librosa.load(tmp, sr=None, mono=True)
        if sr0 != SAMPLE_RATE:
            raw = librosa.resample(raw, orig_sr=sr0, target_sr=SAMPLE_RATE)
        base = _pad_trim(raw)
        for j in range(AUG_PER_CLIP):
            sf.write(str(AUDIO_DIR / "positive" / f"pos_{pos_idx:05d}.wav"),
                     augment(base, SAMPLE_RATE, pos_idx * 10 + j), SAMPLE_RATE)
            pos_idx += 1
        print(f"  {filename} -> {AUG_PER_CLIP} augmented clips added")
    except Exception as e:
        print(f"  [real] {filename}: {e}")

neg_cnt = len(list((AUDIO_DIR / "negative").glob("*.wav")))
print(f"\nTotal positive clips : {pos_idx}")
print(f"Negative clips       : {neg_cnt}")
print(f"\nBreakdown:")
print(f"  gTTS      : {N_GTTS * AUG_PER_CLIP} clips  ({N_GTTS} base)")
print(f"  edge-tts  : {N_EDGE * AUG_PER_CLIP} clips  ({N_EDGE} base, {len(EDGE_VOICES)} voices)")
print(f"  Coqui TTS : ~{len(sel_spk) * AUG_PER_CLIP} clips  ({len(sel_spk)} speakers)")
print(f"  Real recs : {len(uploaded) * AUG_PER_CLIP} clips  ({len(uploaded)} uploaded files)")

### 4. Visualize Waveforms & MFCC Features

In [ ]:
def plot_pair(path, title, aw, am):
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, duration=CLIP_DURATION)
    if len(audio) < NUM_SAMPLES: audio = np.pad(audio, (0, NUM_SAMPLES-len(audio)))
    mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                 n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH)
    aw.plot(np.linspace(0,CLIP_DURATION,len(audio)), audio, color='steelblue', lw=0.6)
    aw.set(title=title, xlabel='Time (s)', ylabel='Amplitude', xlim=(0,CLIP_DURATION))
    librosa.display.specshow(mfcc, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                             x_axis='time', ax=am, cmap='magma')
    am.set(title='MFCC', xlabel='Time (s)', ylabel='Coeff.')

pvs = sorted((AUDIO_DIR/'positive').glob('*.wav'))
nvs = sorted((AUDIO_DIR/'negative').glob('*.wav'))
fig, ax = plt.subplots(4, 2, figsize=(14,10))
fig.suptitle('Sample Clips: Waveform & MFCC', fontsize=13, fontweight='bold')
plot_pair(str(pvs[0]),  '+Positive: Hey Asterix (aug #0)',  ax[0,0], ax[1,0])
plot_pair(str(nvs[0]),  '-Negative: other word',             ax[0,1], ax[1,1])
plot_pair(str(pvs[5]),  '+Positive: Hey Asterix (aug #5)',  ax[2,0], ax[3,0])
plot_pair(str(nvs[10]), '-Negative: background noise',       ax[2,1], ax[3,1])
plt.tight_layout(); plt.show()

## Feature Extraction

### 5. Extract MFCC Features

**Why MFCC?** Compact (~7840 floats/2s clip vs. 32000 raw samples), robust to recording conditions, efficient FFT/DCT on ARM Cortex-M via CMSIS-DSP.

> **Embedded Note**: Re-implement on MCU using ARM CMSIS-DSP or TF Micro audio preprocessing ops.

In [ ]:
def extract_mfcc(wav_path):
    try:
        audio, _ = librosa.load(str(wav_path), sr=SAMPLE_RATE, duration=CLIP_DURATION, mono=True)
    except Exception: return None
    if len(audio) < NUM_SAMPLES: audio = np.pad(audio, (0, NUM_SAMPLES-len(audio)))
    else: audio = audio[:NUM_SAMPLES]
    audio = np.append(audio[0], audio[1:] - 0.97*audio[:-1])  # pre-emphasis
    mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                 n_fft=N_FFT, hop_length=HOP_LENGTH,
                                 win_length=WIN_LENGTH, n_mels=N_MELS).T
    if mfcc.shape[0] < TIME_FRAMES:
        mfcc = np.pad(mfcc, ((0, TIME_FRAMES-mfcc.shape[0]), (0,0)))
    return mfcc[:TIME_FRAMES].astype(np.float32)


def build_dataset(audio_dir):
    Xl, yl = [], []
    for lbl, sub in [(1,'positive'),(0,'negative')]:
        for fp in tqdm(list((audio_dir/sub).glob('*.wav')), desc=f'Extracting {sub}'):
            f = extract_mfcc(fp)
            if f is not None: Xl.append(f); yl.append(lbl)
    return np.array(Xl, dtype=np.float32), np.array(yl, dtype=np.int32)


X_raw, y_raw = build_dataset(AUDIO_DIR)
print(f'Dataset: X={X_raw.shape}, y={y_raw.shape}')
print(f'Balance: {np.bincount(y_raw)} [neg, pos]')

### 6. Normalize Features

Compute mean/std from **training set only**. Store for MCU firmware!

In [ ]:
X = X_raw[..., np.newaxis]  # (N,H,W) -> (N,H,W,1)
y = y_raw

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=VAL_SPLIT+TEST_SPLIT,
                                             random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5,
                                                 random_state=SEED, stratify=y_tmp)

FEAT_MEAN = X_tr.mean()
FEAT_STD  = X_tr.std() + 1e-8
X_train = (X_tr  - FEAT_MEAN) / FEAT_STD
X_val   = (X_val - FEAT_MEAN) / FEAT_STD
X_test  = (X_test - FEAT_MEAN) / FEAT_STD

print(f'Train={X_train.shape}  Val={X_val.shape}  Test={X_test.shape}')
print(f'FEAT_MEAN = {FEAT_MEAN:.6f}')
print(f'FEAT_STD  = {FEAT_STD:.6f}')
print('  Store these in firmware as kFeatMean / kFeatStd!')

np.save(MODELS_DIR+'feat_mean.npy', np.array([FEAT_MEAN]))
np.save(MODELS_DIR+'feat_std.npy',  np.array([FEAT_STD]))

## Build Model

### 7. DS-CNN Architecture

**Depthwise Separable CNN** reduces MACs by ~8-9x vs standard Conv2D. Used in Google MobileNet & KWS research.

Architecture: `Conv2D -> [DW-Conv+BN+ReLU+PW-Conv+BN+ReLU] x4 -> GlobalAvgPool -> Dropout -> Dense(2) -> Softmax`

> Zhang et al., 'Hello Edge: KWS on Microcontrollers', arXiv:1711.07128

In [ ]:
def build_ds_cnn(input_shape, n_cls=2, filters=64, n_blocks=4, drop=0.25):
    inp = keras.Input(shape=input_shape, name='mfcc_input')
    x = layers.Conv2D(filters, (3,3), padding='same', use_bias=False, name='conv1')(inp)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.ReLU(name='relu1')(x)
    for k in range(n_blocks):
        b = f'ds{k+1}'
        x = layers.DepthwiseConv2D((3,3), padding='same', use_bias=False, name=f'{b}_dw')(x)
        x = layers.BatchNormalization(name=f'{b}_dw_bn')(x)
        x = layers.ReLU(name=f'{b}_dw_relu')(x)
        x = layers.Conv2D(filters, (1,1), padding='same', use_bias=False, name=f'{b}_pw')(x)
        x = layers.BatchNormalization(name=f'{b}_pw_bn')(x)
        x = layers.ReLU(name=f'{b}_pw_relu')(x)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(drop, name='dropout')(x)
    out = layers.Dense(n_cls, activation='softmax', name='output')(x)
    return keras.Model(inp, out, name='DS_CNN_KWS')


model = build_ds_cnn(INPUT_SHAPE)
model.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## Train Model

### 8. Training with Callbacks

In [ ]:
cbs = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=10,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=5, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(MODELS_DIR+'best_model.keras',
                              monitor='val_accuracy', save_best_only=True),
]

cw = dict(enumerate(compute_class_weight('balanced',
          classes=np.unique(y_train), y=y_train)))
print(f'Class weights: {cw}')

history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    batch_size=BATCH_SIZE, epochs=EPOCHS,
                    class_weight=cw, callbacks=cbs, verbose=2)
print('Training complete.')

### 9. Plot Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History -- DS-CNN Wake Word', fontsize=13, fontweight='bold')
ep = range(1, len(history.history['loss'])+1)
ax1.plot(ep, history.history['loss'],     'b-o', ms=3, label='Train')
ax1.plot(ep, history.history['val_loss'], 'r-o', ms=3, label='Val')
ax1.set(title='Loss', xlabel='Epoch', ylabel='Cross-Entropy')
ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(ep, history.history['accuracy'],     'b-o', ms=3, label='Train')
ax2.plot(ep, history.history['val_accuracy'], 'r-o', ms=3, label='Val')
ax2.set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy', ylim=(0,1.05))
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Evaluate Model

### 10. Test Set Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss={test_loss:.4f}  Accuracy={test_acc:.4f} ({test_acc*100:.1f}%)')
y_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
print(classification_report(y_test, y_pred, target_names=['Negative','Hey Asterix']))

### 11. Confusion Matrix & ROC Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Test Set Evaluation', fontsize=13, fontweight='bold')

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Negative','Hey Asterix'],
            yticklabels=['Negative','Hey Asterix'])
ax1.set(title='Confusion Matrix', xlabel='Predicted', ylabel='Actual')
tn, fp, fn, tp = cm.ravel()
print(f'FPR={fp/(fp+tn+1e-9):.3f}  FNR={fn/(fn+tp+1e-9):.3f}')

fpr_r, tpr_r, _ = roc_curve(y_test, y_prob[:,1])
ra = auc(fpr_r, tpr_r)
ax2.plot(fpr_r, tpr_r, 'b-', lw=2, label=f'AUC={ra:.3f}')
ax2.plot([0,1],[0,1],'k--',lw=1,label='Random')
ax2.set(title='ROC Curve', xlabel='FPR', ylabel='TPR')
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### 12. Detection Threshold Analysis

**Higher threshold** -> fewer false wakes, more misses. **Lower** -> fewer misses, more false wakes.
Expose as `constexpr float kDetectThreshold` in firmware.

In [ ]:
thrs = np.linspace(0.1, 0.99, 90)
rows = []
for t in thrs:
    yp = (y_prob[:,1] >= t).astype(int)
    cm_t = confusion_matrix(y_test, yp, labels=[0,1])
    tn_t,fp_t,fn_t,tp_t = cm_t.ravel()
    rows.append({'threshold':t, 'FPR':fp_t/(fp_t+tn_t+1e-9),
                 'FNR':fn_t/(fn_t+tp_t+1e-9), 'Acc':(tp_t+tn_t)/len(y_test)})
df = pd.DataFrame(rows); df['err'] = df['FPR']+df['FNR']
opt = df.loc[df['err'].idxmin()]

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(df['threshold'], df['FPR'], 'r-', label='FPR (false wake)')
ax.plot(df['threshold'], df['FNR'], 'b-', label='FNR (missed wake)')
ax.plot(df['threshold'], df['Acc'], 'g--', label='Accuracy')
ax.axvline(0.5, color='gray', ls=':', label='Default (0.5)')
ax.axvline(opt['threshold'], color='purple', ls='--', label=f'Optimal ({opt["threshold"]:.2f})')
ax.set(title='Threshold vs Error Rates', xlabel='Threshold', ylabel='Rate')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Optimal threshold={opt["threshold"]:.2f}  FPR={opt["FPR"]:.3f}  FNR={opt["FNR"]:.3f}')

## Convert to TFLite

### 13. Export Float32 TFLite

In [ ]:
model.save(MODEL_SAVED)
conv = tf.lite.TFLiteConverter.from_saved_model(MODEL_SAVED)
tfl_f32 = conv.convert()
with open(MODEL_TFLITE,'wb') as f: f.write(tfl_f32)
f32_kb = os.path.getsize(MODEL_TFLITE)/1024
print(f'Float32 TFLite: {f32_kb:.1f} KB -> {MODEL_TFLITE}')

### 14. Post-Training INT8 Quantization

Converts all weights **and activations** to INT8 for ~4x size reduction. Required for CMSIS-NN acceleration.

> Provide 100-500 representative samples (unlabeled) for calibration.

In [ ]:
N_REPR = min(500, len(X_train))
repr_d = X_train[np.random.choice(len(X_train), N_REPR, replace=False)].astype(np.float32)

def rep_ds():
    for i in range(N_REPR): yield [repr_d[i:i+1]]

c8 = tf.lite.TFLiteConverter.from_saved_model(MODEL_SAVED)
c8.optimizations = [tf.lite.Optimize.DEFAULT]
c8.representative_dataset = rep_ds
c8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
c8.inference_input_type  = tf.int8
c8.inference_output_type = tf.int8

print('Converting INT8 ...')
tfl_i8 = c8.convert()
with open(MODEL_INT8,'wb') as f: f.write(tfl_i8)
i8_kb = os.path.getsize(MODEL_INT8)/1024
print(f'INT8 TFLite: {i8_kb:.1f} KB  Compression: {f32_kb/i8_kb:.1f}x -> {MODEL_INT8}')

### 15. Validate Quantized Accuracy

In [ ]:
def run_tfl(model_bytes, X):
    interp = litert.Interpreter(model_content=model_bytes)
    interp.allocate_tensors()
    id_ = interp.get_input_details()[0]
    od_ = interp.get_output_details()[0]
    is8 = id_['dtype'] == np.int8
    preds = []
    for i in range(len(X)):
        s = X[i:i+1].astype(np.float32)
        if is8:
            sc,zp = id_['quantization']; s = (s/sc+zp).astype(np.int8)
        interp.set_tensor(id_['index'], s); interp.invoke()
        o = interp.get_tensor(od_['index'])
        if is8:
            osc,ozp = od_['quantization']; o = (o.astype(np.float32)-ozp)*osc
        preds.append(np.argmax(o))
    return np.array(preds)

with open(MODEL_TFLITE,'rb') as f: f32b = f.read()
with open(MODEL_INT8,'rb')   as f: i8b  = f.read()

a32 = np.mean(run_tfl(f32b, X_test) == y_test)
a8  = np.mean(run_tfl(i8b,  X_test) == y_test)
print(f'Float32 TFLite: {a32:.4f} ({a32*100:.1f}%)')
print(f'INT8    TFLite: {a8:.4f}  ({a8*100:.1f}%)')
print(f'Accuracy drop : {(a32-a8)*100:.2f}%')

## Generate C Array

### 16. xxd -i Conversion

In [ ]:
!xxd -i {MODEL_INT8} > {MODEL_CC}
!head -20 {MODEL_CC}
print(f'\nC array -> {MODEL_CC}')

### 17. Patch C Array (`alignas`, `const`, symbol name)

TFLite Micro needs: `const` (stored in Flash), 8-byte aligned (FlatBuffer requirement).

In [ ]:
with open(MODEL_CC,'r') as f: cc = f.read()
cc = re.sub(r'unsigned char \S+\[\]',
            'alignas(8) const unsigned char g_hey_asterix_model_data[]', cc)
cc = re.sub(r'unsigned int \S+_len',
            'const unsigned int g_hey_asterix_model_data_size', cc)
header = '// Auto-generated -- DO NOT EDIT MANUALLY\n'\
         '// Wake word: Hey Asterix -- INT8 DS-CNN\n'\
         '#include <cstdint>\n\n'
with open(MODEL_CC,'w') as f: f.write(header+cc)
print(f'Patched -> {MODEL_CC}')
with open(MODEL_CC) as f: print(''.join(f.readlines()[:20]))

## Firmware Integration

### 18. Inspect Ops (for MicroMutableOpResolver)

Also open `model_int8.tflite` at https://netron.app/

In [ ]:
tf.lite.experimental.Analyzer.analyze(model_path=MODEL_INT8)

### 19. Tensor Arena Sizing

In [ ]:
ia = litert.Interpreter(model_path=MODEL_INT8)
ia.allocate_tensors()
total = 0
for d in ia.get_tensor_details():
    sz = int(np.prod(d['shape']))*np.dtype(d['dtype']).itemsize
    total += sz
    if sz > 256: print(f"  {d['name']:<40s} {d['shape']}  {sz/1024:.2f} KB")
print(f'\nTotal activation memory : {total/1024:.1f} KB')
print(f'Recommended arena size  : {int(total*1.5/1024)} KB (1.5x TFLM overhead)')

### 20. Firmware C++ Template

In [ ]:
fw_tpl = '''
// ====================================================
// Hey Asterix Wake Word Detector  (C++ / TFLite Micro)
// Target: ESP32-S3 / STM32F7
// ====================================================
#include "tensorflow/lite/micro/micro_interpreter.h"
#include "tensorflow/lite/micro/micro_mutable_op_resolver.h"
#include "tensorflow/lite/schema/schema_generated.h"
#include "model.cc"   // g_hey_asterix_model_data[]

constexpr int   kSampleRate      = {sr};
constexpr int   kNumMfcc         = {nm};
constexpr int   kTimeFrames      = {tf};
constexpr float kDetectThreshold = 0.70f;
constexpr float kFeatMean        = {mean:.6f}f;
constexpr float kFeatStd         = {std:.6f}f;

constexpr int  kArenaSize = 80*1024;  // reduce if AllocateTensors() fails
static uint8_t tensor_arena[kArenaSize];

using Resolver = tflite::MicroMutableOpResolver<8>;
static Resolver* CreateResolver() {{
    static Resolver r;
    r.AddConv2D(); r.AddDepthwiseConv2D(); r.AddFullyConnected();
    r.AddReshape(); r.AddSoftmax(); r.AddMean();
    r.AddQuantize(); r.AddDequantize();
    return &r;
}}

static tflite::MicroInterpreter* g_interp = nullptr;
void SetupModel() {{
    auto* m = tflite::GetModel(g_hey_asterix_model_data);
    static Resolver* r = CreateResolver();
    static tflite::MicroInterpreter interp(m, *r, tensor_arena, kArenaSize);
    g_interp = &interp;
    TF_LITE_ASSERT(g_interp->AllocateTensors() == kTfLiteOk);
}}

// mfcc[kTimeFrames][kNumMfcc] pre-computed via CMSIS-DSP or TF Micro audio
bool DetectWakeWord(const float mfcc[][{nm}]) {{
    auto* inp = g_interp->input(0);
    float isc = inp->params.scale; int32_t izp = inp->params.zero_point;
    for (int t=0;t<kTimeFrames;++t)
        for (int m=0;m<kNumMfcc;++m) {{
            float v=(mfcc[t][m]-kFeatMean)/kFeatStd;
            int32_t q=(int32_t)(v/isc)+izp;
            q=q<-128?-128:(q>127?127:q);
            inp->data.int8[t*kNumMfcc+m]=(int8_t)q;
        }}
    TF_LITE_ASSERT(g_interp->Invoke()==kTfLiteOk);
    auto* out=g_interp->output(0);
    float p=(out->data.int8[1]-out->params.zero_point)*out->params.scale;
    return p>=kDetectThreshold;
}}
'''

fw = fw_tpl.format(sr=SAMPLE_RATE, nm=N_MFCC, tf=TIME_FRAMES,
                   mean=FEAT_MEAN, std=FEAT_STD)
print(fw)
with open(MODELS_DIR+'firmware_template.cc','w') as f: f.write(fw)

## Summary

### 21. Final Report

In [ ]:
print('='*62)
print('  HEY ASTERIX WAKE WORD MODEL -- TRAINING SUMMARY')
print('='*62)
print(f'  Architecture  : DS-CNN')
print(f'  Input shape   : {INPUT_SHAPE}')
print(f'  Train/Val/Test: {len(X_train)}/{len(X_val)}/{len(X_test)}')
print(f'  Keras acc     : {test_acc*100:.2f}%')
print(f'  Float32 TFL   : {a32*100:.2f}%  ({f32_kb:.1f} KB)')
print(f'  INT8 TFL      : {a8*100:.2f}%   ({i8_kb:.1f} KB)  <- Deploy!')
print(f'  Quant drop    : {(a32-a8)*100:.2f}%')
print(f'  FEAT_MEAN     : {FEAT_MEAN:.6f}  -> kFeatMean in firmware')
print(f'  FEAT_STD      : {FEAT_STD:.6f}  -> kFeatStd  in firmware')
print('='*62)
print('  MCU Deployment Checklist:')
print('  [1] Copy model.cc to firmware project')
print('  [2] Inspect model_int8.tflite at netron.app')
print('  [3] Register only required ops in MicroMutableOpResolver')
print('  [4] Implement MFCC on MCU (CMSIS-DSP or TF Micro audio)')
print('  [5] Apply kFeatMean/kFeatStd normalization before Invoke()')
print('  [6] Start kArenaSize=80KB; shrink until AllocateTensors() passes')
print('  [7] Tune kDetectThreshold (see FPR/FNR plot)')
print('  [8] Record 500+ real utterances -> retrain for production!')